# Motrex Trips Backfill

Run this notebook locally to execute **Motrex - Group Trips to Athi River/Tororo** for `2026-06-15 00:00` through `2026-06-30 23:59` in Wialon batches of `50` vehicles, with the final batch containing `59` vehicles, and store the processed `Outbound`, `Inbound`, and `TAT` rows in Neon.

The notebook reads `WIALON_TOKEN` and `DATABASE_URL` from `.env.local` / `.env` in this project folder.

In [8]:
# If any import below fails, uncomment and run this line once:
# %pip install requests pandas python-dotenv psycopg2-binary openpyxl

import json
import os
import re
import time
from datetime import datetime, timezone, timedelta
from io import BytesIO
from pathlib import Path
from typing import Any

import pandas as pd
import requests

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

try:
    import psycopg2
    from psycopg2.extras import Json, execute_values
except ImportError as exc:
    raise ImportError("Install psycopg2-binary first: %pip install psycopg2-binary") from exc

PROJECT_DIR = Path.cwd()
if load_dotenv:
    load_dotenv(PROJECT_DIR / ".env.local")
    load_dotenv(PROJECT_DIR / ".env")

API_URL = "https://hst-api.wialon.com/wialon/ajax.html"
WIALON_TOKEN = os.getenv("WIALON_TOKEN", "").strip()
DATABASE_URL = os.getenv("MotrexDB", "").strip()

MOTREX_GROUP_ID = 26659458
MOTREX_RESOURCE_ID = 26054231
TRIPS_TEMPLATE_ID = 15
REPORT_TYPE = "trips"

HISTORICAL_START_DATE = "2026-06-15"
HISTORICAL_END_DATE = "2026-06-30"
HISTORICAL_FROM = datetime(2026, 6, 15, 0, 0, 0, tzinfo=timezone(timedelta(hours=3)))
HISTORICAL_TO = datetime(2026, 6, 30, 23, 59, 59, tzinfo=timezone(timedelta(hours=3)))
REPORT_DATE = HISTORICAL_END_DATE

# Leave empty to discover Motrex units through Wialon search. If discovery fails, paste unit IDs here.
UNIT_IDS_OVERRIDE: list[int] = []
BATCH_COOLDOWN_SECONDS = 60

assert WIALON_TOKEN, "WIALON_TOKEN was not found in .env.local/.env"
assert DATABASE_URL, "DATABASE_URL was not found in .env.local/.env"

print("Configured historical Trips window:", HISTORICAL_FROM, "→", HISTORICAL_TO)
print("Project folder:", PROJECT_DIR)

Configured historical Trips window: 2026-06-15 00:00:00+03:00 → 2026-06-30 23:59:59+03:00
Project folder: c:\Users\YOGA\OneDrive - ControlTech Limited\ControlTech\Wialon\Motrex\motrex-fleet-insights


In [9]:
def wialon_call(svc: str, params: dict[str, Any], sid: str | None = None) -> Any:
    data = {"svc": svc, "params": json.dumps(params)}
    if sid:
        data["sid"] = sid
    response = requests.post(API_URL, data=data, timeout=180)
    response.raise_for_status()
    payload = response.json()
    if isinstance(payload, dict) and payload.get("error") not in (None, 0):
        raise RuntimeError(f"Wialon error {payload.get('error')}: {payload}")
    return payload


def wialon_login() -> str:
    login = wialon_call("token/login", {"token": WIALON_TOKEN})
    sid = login.get("eid")
    if not sid:
        raise RuntimeError(f"Wialon login failed: {login}")
    wialon_call("render/set_locale", {"tzOffset": 10800, "language": "en", "formatDate": "%d.%m.%Y %H:%M:%S"}, sid)
    return sid


def wialon_logout(sid: str) -> None:
    try:
        wialon_call("core/logout", {}, sid)
    except Exception:
        pass


def get_report_template(sid: str) -> dict[str, Any]:
    templates = wialon_call("report/get_report_data", {"itemId": MOTREX_RESOURCE_ID, "col": [TRIPS_TEMPLATE_ID]}, sid)
    if not templates:
        raise RuntimeError("Trips template was not found in Wialon")
    return templates[0]


def wait_for_remote_report_result(sid: str, batch_no: int, max_wait_minutes: int = 60) -> dict[str, Any]:
    attempts = max_wait_minutes * 12
    for attempt in range(attempts):
        status = wialon_call("report/get_report_status", {}, sid)
        code = status.get("status") if isinstance(status, dict) else None
        if code == 4:
            print(f"Batch {batch_no}: remote report ready")
            return wialon_call("report/apply_report_result", {}, sid)
        if code in (8, 16):
            raise RuntimeError(f"Remote report failed with status {code}: {status}")
        if attempt > 0 and attempt % 12 == 0:
            print(f"Batch {batch_no}: remote report still processing ({attempt // 12}m)")
        time.sleep(5)
    raise RuntimeError(f"Remote report timed out after {max_wait_minutes} minutes")


def discover_motrex_unit_ids(sid: str) -> list[int]:
    if UNIT_IDS_OVERRIDE:
        return sorted(set(int(v) for v in UNIT_IDS_OVERRIDE))

    # Search all accessible units and keep Motrex units. This mirrors the local API-testing style and avoids group lookup failures.
    payload = wialon_call(
        "core/search_items",
        {
            "spec": {
                "itemsType": "avl_unit",
                "propName": "sys_name",
                "propValueMask": "*",
                "sortType": "sys_name",
            },
            "force": 1,
            "flags": 1,
            "from": 0,
            "to": 10000,
        },
        sid,
    )
    units = payload.get("items", []) if isinstance(payload, dict) else []
    ids = [int(u["id"]) for u in units if re.match(r"^Motrex\s*-", str(u.get("nm", "")), re.I) and u.get("id")]
    if not ids:
        raise RuntimeError("No Motrex units found. If Wialon search is restricted, paste unit IDs into UNIT_IDS_OVERRIDE and rerun.")
    return sorted(set(ids))


def split_50_vehicle_batches(unit_ids: list[int]) -> list[list[int]]:
    batches = [unit_ids[i * 50 : i * 50 + 50] for i in range(11)]
    batches.append(unit_ids[550:])
    return [b for b in batches if b]


def cell_text(cell: Any) -> str:
    if isinstance(cell, dict):
        return str(cell.get("t", "") or "")
    return str(cell or "")


def fetch_table_rows(sid: str, tables: list[dict[str, Any]], table_index: int) -> tuple[list[str], list[dict[str, str]]]:
    table = tables[table_index]
    headers = table.get("header", []) or []
    row_count = int(table.get("rows", 0) or 0)
    if row_count <= 0:
        return headers, []

    all_rows = []
    for row_index in range(row_count):
        sub = wialon_call(
            "report/get_result_subrows",
            {"tableIndex": table_index, "rowIndex": row_index, "colIndex": 0, "indexFrom": 0, "indexTo": 1000},
            sid,
        )
        if isinstance(sub, list):
            all_rows.extend(sub)

    if not all_rows:
        rows = wialon_call(
            "report/get_result_rows",
            {"tableIndex": table_index, "indexFrom": 0, "indexTo": max(row_count - 1, 0)},
            sid,
        )
        if isinstance(rows, list):
            all_rows = rows

    clean_rows: list[dict[str, str]] = []
    for row in all_rows:
        cells = row.get("c", []) if isinstance(row, dict) else []
        out = {}
        for i, header in enumerate(headers):
            out[str(header or f"col_{i}").strip()] = cell_text(cells[i] if i < len(cells) else "")
        clean_rows.append(out)
    return headers, clean_rows


def execute_single_vehicle_trips(sid: str, unit_id: int) -> list[dict[str, str]]:
    from_ts = int(HISTORICAL_FROM.timestamp())
    to_ts = int(HISTORICAL_TO.timestamp())
    print(f"Single vehicle: executing direct stored-template report ({unit_id})")
    wialon_call("report/cleanup_result", {}, sid)
    result = wialon_call(
        "report/exec_report",
        {
            "reportResourceId": MOTREX_RESOURCE_ID,
            "reportTemplateId": TRIPS_TEMPLATE_ID,
            "reportObjectId": unit_id,
            "reportObjectSecId": 0,
            "interval": {"from": from_ts, "to": to_ts, "flags": 0},
        },
        sid,
    )
    tables = result.get("reportResult", {}).get("tables", [])
    print("Single vehicle tables:", [{"index": i, "label": t.get("label"), "rows": t.get("rows")} for i, t in enumerate(tables)])
    best_rows: list[dict[str, str]] = []
    for table_index, table in enumerate(tables):
        if int(table.get("rows", 0) or 0) <= 0:
            continue
        _, rows = fetch_table_rows(sid, tables, table_index)
        best_rows.extend(rows)
    print(f"Single vehicle: fetched {len(best_rows)} raw rows")
    return best_rows


def execute_trips_batch(sid: str, template: dict[str, Any], batch: list[int], batch_no: int) -> list[dict[str, str]]:
    # Wialon is fast and reliable when this Trips template is executed per vehicle.
    # The 50-vehicle batches are kept for progress tracking and controlled pacing.
    batch_rows: list[dict[str, str]] = []
    failed_units: list[dict[str, Any]] = []
    print(f"Batch {batch_no}: executing {len(batch)} vehicles one by one with direct stored-template reports")

    for vehicle_index, unit_id in enumerate(batch, start=1):
        try:
            rows = execute_single_vehicle_trips(sid, unit_id)
            batch_rows.extend(rows)
            print(f"Batch {batch_no}: vehicle {vehicle_index}/{len(batch)} unit {unit_id} fetched {len(rows)} raw rows")
        except Exception as exc:
            failed_units.append({"unitId": unit_id, "error": str(exc)})
            print(f"Batch {batch_no}: vehicle {vehicle_index}/{len(batch)} unit {unit_id} failed: {exc}")
        time.sleep(1)

    if failed_units:
        print(f"Batch {batch_no}: {len(failed_units)} vehicles failed and were skipped:", failed_units)
    print(f"Batch {batch_no}: fetched {len(batch_rows)} raw rows total")
    return batch_rows

In [10]:
def registration_label(value: Any) -> str:
    return re.sub(r"^Motrex\s*-\s*", "", str(value or "").strip(), flags=re.I).strip()


def pick_key(row: dict[str, Any], patterns: list[str]) -> str | None:
    for key in row.keys():
        lower = key.lower()
        if any(re.search(pattern, lower, re.I) for pattern in patterns):
            return key
    return None


def parse_dt(value: Any) -> pd.Timestamp | None:
    raw = str(value or "").strip()
    if not raw or raw == "-----":
        return None
    dayfirst = not bool(re.match(r"^\d{4}-\d{2}-\d{2}", raw))
    dt = pd.to_datetime(raw, dayfirst=dayfirst, errors="coerce")
    if pd.isna(dt):
        return None
    return dt


WIALON_TRIP_COLUMNS = [
    "Grouping",
    "Trip",
    "Trip from",
    "Trip to",
    "Beginning",
    "End",
    "Mileage",
    "Consumed by AbsFCS",
    "Avg consumption by AbsFCS",
    "Trip duration",
    "Total time",
    "Parkings duration",
    "Avg speed",
    "Max speed",
    "Initial fuel level",
    "Final fuel level",
    "Count",
]


def preserve_wialon_trip_columns(row: dict[str, Any]) -> dict[str, Any]:
    return {column: row.get(column, "") for column in WIALON_TRIP_COLUMNS}


def endpoint_for_geofence(geofence: Any) -> str | None:
    text = str(geofence or "").lower()
    if "tororo" in text:
        return "tororo"
    if "athi" in text or "arthi" in text or "mombasa" in text or "vipingo" in text:
        return "athi"
    return None


def endpoint_label(endpoint: str) -> str:
    return "Tororo" if endpoint == "tororo" else "Athi River"


def duration_label(end_value: Any, start_value: Any) -> str:
    end = parse_dt(end_value)
    start = parse_dt(start_value)
    if end is None or start is None or end <= start:
        return ""
    total_minutes = int((end - start).total_seconds() // 60)
    days, rem = divmod(total_minutes, 1440)
    hours, minutes = divmod(rem, 60)
    if days:
        return f"{days}d {hours}h {minutes}m"
    if hours:
        return f"{hours}h {minutes}m"
    return f"{minutes}m"


def build_trip_tables(raw_rows: list[dict[str, Any]], report_date: str) -> list[dict[str, Any]]:
    events = []
    for row in raw_rows:
        vehicle_key = pick_key(row, [r"^vehicle$", r"vehicle", r"unit", r"registration", r"plate", r"name", r"grouping"])
        geofence_key = pick_key(row, [r"geofence", r"geozone", r"location", r"place"])
        time_in_key = pick_key(row, [r"time\s*in", r"beginning", r"entry", r"arrival", r"from"])
        time_out_key = pick_key(row, [r"time\s*out", r"end", r"exit", r"departure", r"to"])
        geofence = row.get(geofence_key, "") if geofence_key else ""
        endpoint = endpoint_for_geofence(geofence)
        time_in = row.get(time_in_key, "") if time_in_key else ""
        time_out = row.get(time_out_key, "") if time_out_key else ""
        timestamp = parse_dt(time_in or time_out)
        vehicle = registration_label(row.get(vehicle_key, "") if vehicle_key else "")
        if vehicle and endpoint and timestamp is not None:
            events.append({
                "vehicle": vehicle,
                "endpoint": endpoint,
                "geofence": geofence,
                "time_in": str(time_in or ""),
                "time_out": str(time_out or ""),
                "timestamp": timestamp,
            })

    outbound = []
    inbound = []

    # Trips reports return direct leg rows with Trip from/to and Beginning/End columns.
    for row in raw_rows:
        vehicle_key = pick_key(row, [r"^vehicle$", r"vehicle", r"unit", r"registration", r"plate", r"name", r"grouping"])
        from_key = pick_key(row, [r"^trip\s*from$", r"trip\s*from"])
        to_key = pick_key(row, [r"^trip\s*to$", r"trip\s*to"])
        beginning_key = pick_key(row, [r"^beginning$", r"beginning", r"departure"])
        end_key = pick_key(row, [r"^end$", r"arrival"])
        if not (from_key and to_key and beginning_key and end_key):
            continue
        from_endpoint = endpoint_for_geofence(row.get(from_key, ""))
        to_endpoint = endpoint_for_geofence(row.get(to_key, ""))
        vehicle = registration_label(row.get(vehicle_key, "") if vehicle_key else "")
        if not (vehicle and from_endpoint and to_endpoint and from_endpoint != to_endpoint):
            continue
        departure = str(row.get(beginning_key, "") or "")
        arrival = str(row.get(end_key, "") or "")
        table = "Outbound" if from_endpoint == "athi" and to_endpoint == "tororo" else "Inbound"
        leg = {
            **preserve_wialon_trip_columns(row),
            "Table": table,
            "Vehicle": vehicle,
            "From": endpoint_label(from_endpoint),
            "To": endpoint_label(to_endpoint),
            "Trip Count": row.get("Count", 1) or 1,
            "Departure Time": departure,
            "Arrival Time": arrival,
            "Transit Time": row.get("Trip duration", "") or duration_label(arrival, departure),
            "Parkings duration": row.get("Parkings duration", ""),
            "Total time": row.get("Total time", ""),
            "Report Date": report_date,
        }
        if table == "Outbound":
            outbound.append(leg)
        else:
            inbound.append(leg)

    grouped_events = pd.DataFrame(events).groupby("vehicle") if events else []
    for vehicle, group in grouped_events:
        group = group.sort_values("timestamp")
        previous = None
        for _, event in group.iterrows():
            current = event.to_dict()
            if previous is None:
                previous = current
                continue
            if current["endpoint"] == previous["endpoint"]:
                previous = current
                continue
            departure = previous["time_out"] or previous["time_in"]
            arrival = current["time_in"] or current["time_out"]
            table = "Outbound" if previous["endpoint"] == "athi" and current["endpoint"] == "tororo" else "Inbound"
            leg = {
                "Table": table,
                "Vehicle": vehicle,
                "From": endpoint_label(previous["endpoint"]),
                "To": endpoint_label(current["endpoint"]),
                "Departure Time": departure,
                "Arrival Time": arrival,
                "Transit Time": duration_label(arrival, departure),
                "Trip Count": 1,
                "Report Date": report_date,
            }
            if table == "Outbound":
                outbound.append(leg)
            else:
                inbound.append(leg)
            previous = current

    tat = []
    inbound_by_vehicle: dict[str, list[dict[str, Any]]] = {}
    for leg in inbound:
        inbound_by_vehicle.setdefault(leg["Vehicle"], []).append(leg)

    for leg in outbound:
        arrival = parse_dt(leg["Arrival Time"])
        candidates = []
        for candidate in inbound_by_vehicle.get(leg["Vehicle"], []):
            dep = parse_dt(candidate["Departure Time"])
            if arrival is not None and dep is not None and dep >= arrival:
                candidates.append(candidate)
        candidates.sort(key=lambda r: parse_dt(r["Departure Time"]) or pd.Timestamp.max)
        if not candidates:
            continue
        ret = candidates[0]
        tat.append({
            "Table": "TAT",
            "Vehicle": leg["Vehicle"],
            "Tororo Departure": leg["Departure Time"],
            "Athi River Arrival": leg["Arrival Time"],
            "Athi River Departure": ret["Departure Time"],
            "Tororo Return": ret["Arrival Time"],
            "Outbound Transit": leg["Transit Time"],
            "Time at Athi River": duration_label(ret["Departure Time"], leg["Arrival Time"]),
            "Inbound Transit": ret["Transit Time"],
            "Full Round-Trip TAT": duration_label(ret["Arrival Time"], leg["Departure Time"]),
            "Trip Count": 1,
            "Report Date": report_date,
        })
    return outbound + inbound + tat

In [11]:
def ensure_neon_tables(conn) -> None:
    with conn.cursor() as cur:
        cur.execute(
            """
            CREATE TABLE IF NOT EXISTS motrex_trips (
              id SERIAL PRIMARY KEY,
              week_start DATE NOT NULL,
              week_end DATE NOT NULL,
              trip_type TEXT NOT NULL DEFAULT 'Raw',
              registration_number TEXT NOT NULL,
              vehicle TEXT NOT NULL,
              raw_row JSONB NOT NULL,
              created_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
              updated_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
            );
            """
        )
        cur.execute("ALTER TABLE motrex_trips ADD COLUMN IF NOT EXISTS trip_type TEXT NOT NULL DEFAULT 'Raw';")
        for column_name in [
            "grouping", "trip", "trip_from", "trip_to", "beginning", "end", "mileage",
            "consumed_by_abs_fcs", "avg_consumption_by_abs_fcs", "trip_duration", "total_time",
            "parkings_duration", "avg_speed", "max_speed", "initial_fuel_level", "final_fuel_level", "count",
        ]:
            cur.execute(f'ALTER TABLE motrex_trips ADD COLUMN IF NOT EXISTS "{column_name}" TEXT;')
        cur.execute(
            """
            CREATE TABLE IF NOT EXISTS report_snapshots (
              id SERIAL PRIMARY KEY,
              report_type TEXT NOT NULL,
              report_date DATE NOT NULL,
              payload JSONB NOT NULL,
              raw_meta JSONB,
              created_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
              updated_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
              UNIQUE (report_type, report_date)
            );
            """
        )
    conn.commit()


def upsert_trips_to_neon(processed_rows: list[dict[str, Any]], raw_rows: list[dict[str, Any]], batch_meta: list[dict[str, Any]]) -> None:
    conn = psycopg2.connect(DATABASE_URL)
    try:
        ensure_neon_tables(conn)
        with conn.cursor() as cur:
            cur.execute(
                "DELETE FROM motrex_trips WHERE week_start = %s AND week_end = %s",
                (HISTORICAL_START_DATE, HISTORICAL_END_DATE),
            )
            if processed_rows:
                values = []
                for row in processed_rows:
                    vehicle = registration_label(row.get("Vehicle", "Unknown")) or "Unknown"
                    values.append((
                        HISTORICAL_START_DATE,
                        HISTORICAL_END_DATE,
                        str(row.get("Table", "Raw")),
                        vehicle,
                        vehicle,
                        Json(row),
                    ))
                execute_values(
                    cur,
                    """
                    INSERT INTO motrex_trips
                      (week_start, week_end, trip_type, registration_number, vehicle, raw_row, updated_at)
                    VALUES %s
                    """,
                    values,
                    template="(%s, %s, %s, %s, %s, %s, NOW())",
                )

            payload = {"rows": processed_rows}
            raw_meta = {
                "rowCount": len(processed_rows),
                "rawRowCount": len(raw_rows),
                "from": int(HISTORICAL_FROM.timestamp()),
                "to": int(HISTORICAL_TO.timestamp()),
                "intervalStart": HISTORICAL_START_DATE,
                "intervalEnd": HISTORICAL_END_DATE,
                "weekStart": HISTORICAL_START_DATE,
                "weekEnd": HISTORICAL_END_DATE,
                "batchCount": len(batch_meta),
                "batches": batch_meta,
                "source": "Trips.ipynb",
            }
            cur.execute(
                """
                INSERT INTO report_snapshots (report_type, report_date, payload, raw_meta, updated_at)
                VALUES (%s, %s, %s, %s, NOW())
                ON CONFLICT (report_type, report_date)
                DO UPDATE SET payload = EXCLUDED.payload, raw_meta = EXCLUDED.raw_meta, updated_at = NOW()
                """,
                (REPORT_TYPE, REPORT_DATE, Json(payload), Json(raw_meta)),
            )
        conn.commit()
        print(f"Stored {len(processed_rows)} processed Trips rows in Neon.")
    finally:
        conn.close()

In [12]:
def open_neon_connection():
    conn = psycopg2.connect(DATABASE_URL)
    ensure_neon_tables(conn)
    return conn


def prepare_incremental_trips_run(conn) -> None:
    with conn.cursor() as cur:
        cur.execute(
            "DELETE FROM motrex_trips WHERE week_start = %s AND week_end = %s",
            (HISTORICAL_START_DATE, HISTORICAL_END_DATE),
        )
    conn.commit()
    print(f"Cleared existing Trips rows for {HISTORICAL_START_DATE} -> {HISTORICAL_END_DATE}")


def insert_vehicle_trips_to_neon(conn, processed_rows: list[dict[str, Any]], unit_id: int) -> int:
    vehicle_names = sorted({registration_label(row.get("Vehicle", "Unknown")) or "Unknown" for row in processed_rows})
    with conn.cursor() as cur:
        # Keep each vehicle idempotent, so rerunning a vehicle replaces that vehicle's rows.
        for vehicle in vehicle_names:
            cur.execute(
                """
                DELETE FROM motrex_trips
                WHERE week_start = %s AND week_end = %s AND registration_number = %s
                """,
                (HISTORICAL_START_DATE, HISTORICAL_END_DATE, vehicle),
            )

        if processed_rows:
            values = []
            for row in processed_rows:
                vehicle = registration_label(row.get("Vehicle", "Unknown")) or "Unknown"
                stored_row = dict(row)
                stored_row["Wialon Unit ID"] = unit_id
                values.append((
                    HISTORICAL_START_DATE,
                    HISTORICAL_END_DATE,
                    str(row.get("Table", "Raw")),
                    vehicle,
                    vehicle,
                    str(row.get("Grouping", "") or ""),
                    str(row.get("Trip", "") or ""),
                    str(row.get("Trip from", "") or ""),
                    str(row.get("Trip to", "") or ""),
                    str(row.get("Beginning", "") or ""),
                    str(row.get("End", "") or ""),
                    str(row.get("Mileage", "") or ""),
                    str(row.get("Consumed by AbsFCS", "") or ""),
                    str(row.get("Avg consumption by AbsFCS", "") or ""),
                    str(row.get("Trip duration", "") or ""),
                    str(row.get("Total time", "") or ""),
                    str(row.get("Parkings duration", "") or ""),
                    str(row.get("Avg speed", "") or ""),
                    str(row.get("Max speed", "") or ""),
                    str(row.get("Initial fuel level", "") or ""),
                    str(row.get("Final fuel level", "") or ""),
                    str(row.get("Count", "") or ""),
                    Json(stored_row),
                ))
            execute_values(
                cur,
                """
                INSERT INTO motrex_trips
                  (week_start, week_end, trip_type, registration_number, vehicle,
                   "grouping", "trip", "trip_from", "trip_to", "beginning", "end", "mileage",
                   "consumed_by_abs_fcs", "avg_consumption_by_abs_fcs", "trip_duration",
                   "total_time", "parkings_duration", "avg_speed", "max_speed",
                   "initial_fuel_level", "final_fuel_level", "count", raw_row, updated_at)
                VALUES %s
                """,
                values,
                template="(%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, NOW())",
            )
    conn.commit()
    return len(processed_rows)


def update_incremental_trips_snapshot(conn, batch_meta: list[dict[str, Any]]) -> None:
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT raw_row FROM motrex_trips
            WHERE week_start = %s AND week_end = %s
            ORDER BY id
            """,
            (HISTORICAL_START_DATE, HISTORICAL_END_DATE),
        )
        rows = [record[0] for record in cur.fetchall()]
        raw_meta = {
            "rowCount": len(rows),
            "from": int(HISTORICAL_FROM.timestamp()),
            "to": int(HISTORICAL_TO.timestamp()),
            "intervalStart": HISTORICAL_START_DATE,
            "intervalEnd": HISTORICAL_END_DATE,
            "weekStart": HISTORICAL_START_DATE,
            "weekEnd": HISTORICAL_END_DATE,
            "batchCount": len(batch_meta),
            "batches": batch_meta,
            "source": "Trips.ipynb incremental per-vehicle run",
        }
        cur.execute(
            """
            INSERT INTO report_snapshots (report_type, report_date, payload, raw_meta, updated_at)
            VALUES (%s, %s, %s, %s, NOW())
            ON CONFLICT (report_type, report_date)
            DO UPDATE SET payload = EXCLUDED.payload, raw_meta = EXCLUDED.raw_meta, updated_at = NOW()
            """,
            (REPORT_TYPE, REPORT_DATE, Json({"rows": rows}), Json(raw_meta)),
        )
    conn.commit()
    print(f"Updated report snapshot with {len(rows)} stored Trips rows.")

## Single Vehicle Test

Run this cell first to validate the Trips report for one vehicle only: `Motrex - KBT 365P`. If this works, then continue to the full 50-vehicle batch execution cell below.

In [13]:
TEST_VEHICLE_NAME = "Motrex - KBT 365P"

sid = wialon_login()
try:
    print("Logged in to Wialon")

    units_payload = wialon_call(
        "core/search_items",
        {
            "spec": {
                "itemsType": "avl_unit",
                "propName": "sys_name",
                "propValueMask": TEST_VEHICLE_NAME,
                "sortType": "sys_name",
            },
            "force": 1,
            "flags": 1,
            "from": 0,
            "to": 10,
        },
        sid,
    )
    items = units_payload.get("items", []) if isinstance(units_payload, dict) else []
    exact = [unit for unit in items if unit.get("nm") == TEST_VEHICLE_NAME]
    if not exact and not items:
        raise RuntimeError(f"Could not find {TEST_VEHICLE_NAME} in Wialon")
    test_unit_id = int((exact or items)[0]["id"])
    print(f"Resolved {TEST_VEHICLE_NAME} -> {test_unit_id}")

    raw_rows = execute_single_vehicle_trips(sid, test_unit_id)
    print("Raw rows:", len(raw_rows))

    processed_rows = build_trip_tables(raw_rows, REPORT_DATE)
    print("Processed rows:", len(processed_rows))

    if processed_rows:
        test_df = pd.DataFrame(processed_rows)
        display(test_df.groupby("Table").size().rename("rows"))
        display(test_df.head(20))
    else:
        print("No Outbound/Inbound/TAT rows were derived for the test vehicle. Raw sample:")
        display(pd.DataFrame(raw_rows).head(20))
finally:
    wialon_logout(sid)
    print("Logged out of Wialon")

Logged in to Wialon
Resolved Motrex - KBT 365P -> 27492530
Single vehicle: executing direct stored-template report (27492530)
Single vehicle tables: [{'index': 0, 'label': 'Motrex - Tororo - Trips between geofences', 'rows': 1}, {'index': 1, 'label': 'Tororo - Motrex - Trips between geofences', 'rows': 1}]
Single vehicle: fetched 2 raw rows
Raw rows: 2
Processed rows: 2


Table
Inbound     1
Outbound    1
Name: rows, dtype: int64

,Grouping,Trip,Trip from,Trip to,Beginning,End,Mileage,Consumed by AbsFCS,Avg consumption by AbsFCS,Trip duration,...,Count,Table,Vehicle,From,To,Trip Count,Departure Time,Arrival Time,Transit Time,Report Date
0,Motrex - KBT 365P,Tororo Cement(Uganda) - Mombasa Cement(Vipingo...,Tororo Cement(Uganda),Mombasa Cement(Vipingo Area),2026-06-22 19:43:09,2026-06-25 11:36:54,1014 km,309 l,30 l/100 km,2 days 15:53:45,...,1,Outbound,KBT 365P,Tororo,Athi River,1,2026-06-22 19:43:09,2026-06-25 11:36:54,2 days 15:53:45,2026-06-30
1,Motrex - KBT 365P,Mombasa Cement(Vipingo Area) - Tororo Cement(U...,Mombasa Cement(Vipingo Area),Tororo Cement(Uganda),2026-06-19 15:56:04,2026-06-30 05:44:51,1921 km,1057 l,55 l/100 km,6 days 13:48:37,...,2,Inbound,KBT 365P,Athi River,Tororo,2,2026-06-19 15:56:04,2026-06-30 05:44:51,6 days 13:48:37,2026-06-30


Logged out of Wialon


In [ ]:
sid = wialon_login()
neon_conn = open_neon_connection()
try:
    print("Logged in to Wialon")
    unit_ids = discover_motrex_unit_ids(sid)
    batches = split_50_vehicle_batches(unit_ids)
    print("Resolved units:", len(unit_ids))
    print("Trips unit batches:", " + ".join(str(len(batch)) for batch in batches), "vehicles")

    prepare_incremental_trips_run(neon_conn)

    batch_meta: list[dict[str, Any]] = []
    total_raw_rows = 0
    total_processed_rows = 0
    total_vehicles_done = 0
    failed_units: list[dict[str, Any]] = []

    for batch_index, batch in enumerate(batches, start=1):
        batch_raw_rows = 0
        batch_processed_rows = 0
        print(f"Batch {batch_index}: running {len(batch)} vehicles one at a time")

        for vehicle_index, unit_id in enumerate(batch, start=1):
            try:
                raw_rows = execute_single_vehicle_trips(sid, unit_id)
                processed_rows = build_trip_tables(raw_rows, REPORT_DATE)
                stored_count = insert_vehicle_trips_to_neon(neon_conn, processed_rows, unit_id)

                total_raw_rows += len(raw_rows)
                total_processed_rows += stored_count
                batch_raw_rows += len(raw_rows)
                batch_processed_rows += stored_count
                total_vehicles_done += 1

                print(
                    f"Batch {batch_index}: vehicle {vehicle_index}/{len(batch)} "
                    f"unit {unit_id} stored {stored_count} rows in Neon "
                    f"({total_vehicles_done}/{len(unit_ids)} vehicles done)"
                )
            except Exception as exc:
                failed_units.append({"unitId": unit_id, "error": str(exc)})
                total_vehicles_done += 1
                print(
                    f"Batch {batch_index}: vehicle {vehicle_index}/{len(batch)} "
                    f"unit {unit_id} failed: {exc} "
                    f"({total_vehicles_done}/{len(unit_ids)} vehicles done)"
                )
            time.sleep(1)

        batch_meta.append({
            "batch": batch_index,
            "vehicles": len(batch),
            "rawRows": batch_raw_rows,
            "processedRows": batch_processed_rows,
        })
        update_incremental_trips_snapshot(neon_conn, batch_meta)
        print(f"Batch {batch_index}: stored {batch_processed_rows} processed rows from {batch_raw_rows} raw rows")
        if total_vehicles_done < len(unit_ids):
            print(f"Cooling down for {BATCH_COOLDOWN_SECONDS} seconds after {total_vehicles_done} vehicles...")
            time.sleep(BATCH_COOLDOWN_SECONDS)

    print("Total raw rows:", total_raw_rows)
    print("Total processed rows stored in Neon:", total_processed_rows)
    if failed_units:
        print("Failed units:", failed_units)

    update_incremental_trips_snapshot(neon_conn, batch_meta)
finally:
    neon_conn.close()
    wialon_logout(sid)
    print("Logged out of Wialon")

Logged in to Wialon
Resolved units: 609
Trips unit batches: 50 + 50 + 50 + 50 + 50 + 50 + 50 + 50 + 50 + 50 + 50 + 59 vehicles
Cleared existing Trips rows for 2026-06-15 -> 2026-06-30
Batch 1: running 50 vehicles one at a time
Single vehicle: executing direct stored-template report (26056137)
Single vehicle tables: []
Single vehicle: fetched 0 raw rows
Batch 1: vehicle 1/50 unit 26056137 stored 0 rows in Neon (1/609 vehicles done)
Single vehicle: executing direct stored-template report (26056149)
Single vehicle tables: [{'index': 0, 'label': 'Motrex - Tororo - Trips between geofences', 'rows': 1}, {'index': 1, 'label': 'Tororo - Motrex - Trips between geofences', 'rows': 1}]
Single vehicle: fetched 2 raw rows
Batch 1: vehicle 2/50 unit 26056149 stored 2 rows in Neon (2/609 vehicles done)
Single vehicle: executing direct stored-template report (26056153)
Single vehicle tables: [{'index': 0, 'label': 'Tororo - Motrex - Trips between geofences', 'rows': 1}]
Single vehicle: fetched 1 raw r